In [2]:

import pandas as pd
import numpy as np

# Load flood dataset
flood_df = pd.read_csv("../../../data/disaster_data/floods/major_floods_2000_2026.csv")

# Convert dates
flood_df["start_date"] = pd.to_datetime(flood_df["start_date"])
flood_df["end_date"] = pd.to_datetime(flood_df["end_date"])

# Ensure numeric columns
numeric_cols = [
    "total_damage_usd",
    "total_deaths",
    "total_injuries",
    "duration_days"
]

for col in numeric_cols:
    flood_df[col] = pd.to_numeric(
        flood_df[col], errors="coerce"
    )

# Remove duplicate flood IDs
flood_df = flood_df.drop_duplicates(subset=["flood_id"])

print("Total flood events:", len(flood_df))
display(flood_df.head())


Total flood events: 158


,flood_id,episode_id,start_date,end_date,states,flood_types,total_damage_usd,total_deaths,total_injuries,affected_records,duration_days
0,FLOOD_119753,119753,2017-08-25 23:45:00,2017-08-31 12:00:00,TEXAS,Flash Flood,3.825064e+10,55,0,72,5.510417
1,FLOOD_68471,68471,2012-10-28 18:00:00,2012-10-31 17:00:00,NEW JERSEY,"Coastal Flood, Flood",2.095000e+10,5,0,25,2.958333
2,FLOOD_120357,120357,2017-09-20 00:00:00,2017-09-22 08:00:00,PUERTO RICO,Flash Flood,1.825875e+10,20,0,41,2.333333
3,FLOOD_109514,109514,2016-08-11 09:00:00,2016-08-27 16:00:00,LOUISIANA,"Flash Flood, Flood",6.982976e+09,12,0,23,16.291667
4,FLOOD_119746,119746,2017-08-26 12:30:00,2017-08-31 23:59:00,TEXAS,"Flash Flood, Flood",5.290000e+09,18,1,8,5.478472


In [3]:

N = 10

top_floods = (
    flood_df
    .sort_values("total_damage_usd", ascending=False)
    .head(N)
    .copy()
)

top_floods["damage_billions"] = (
    top_floods["total_damage_usd"] / 1e9
)

display(
    top_floods[
        ["flood_id", "start_date", "states",
         "flood_types", "damage_billions"]
    ]
)


,flood_id,start_date,states,flood_types,damage_billions
0,FLOOD_119753,2017-08-25 23:45:00,TEXAS,Flash Flood,38.250641
1,FLOOD_68471,2012-10-28 18:00:00,NEW JERSEY,"Coastal Flood, Flood",20.950000
2,FLOOD_120357,2017-09-20 00:00:00,PUERTO RICO,Flash Flood,18.258750
3,FLOOD_109514,2016-08-11 09:00:00,LOUISIANA,"Flash Flood, Flood",6.982976
4,FLOOD_119746,2017-08-26 12:30:00,TEXAS,"Flash Flood, Flood",5.290000
5,FLOOD_53127,2011-05-01 00:00:00,TENNESSEE,Flood,2.003000
6,FLOOD_89491,2014-08-11 15:00:00,MICHIGAN,"Flash Flood, Flood",1.800000
7,FLOOD_196694,2024-09-25 18:00:00,NORTH CAROLINA,"Flash Flood, Flood",1.585327
8,FLOOD_37606,2010-05-01 10:00:00,TENNESSEE,"Flash Flood, Flood",1.569464
9,FLOOD_110040,2016-08-14 06:00:00,LOUISIANA,Flood,1.400000


In [6]:

market_df = pd.read_csv(
    "../../../data/stock_data/market_raw.csv"
)


market_df["Date"] = pd.to_datetime(market_df["Date"])

market_df = (
    market_df
    .sort_values("Date")
    .drop_duplicates(subset=["Date"])
    .reset_index(drop=True)
)

sectors = ["SPY", "XLE", "XLU", "XLF", "XLI"]
windows = [1, 5, 14]

top_floods = pd.merge_asof(
    top_floods.sort_values("start_date"),
    market_df[["Date"]].sort_values("Date"),
    left_on="start_date",
    right_on="Date",
    direction="forward"
)

top_floods = top_floods.rename(
    columns={"Date": "event_trading_date"}
)

display(
    top_floods[
        ["flood_id", "start_date", "event_trading_date"]
    ]
)


,flood_id,start_date,event_trading_date
0,FLOOD_37606,2010-05-01 10:00:00,2010-05-03
1,FLOOD_53127,2011-05-01 00:00:00,2011-05-02
2,FLOOD_68471,2012-10-28 18:00:00,2012-10-31
3,FLOOD_89491,2014-08-11 15:00:00,2014-08-12
4,FLOOD_109514,2016-08-11 09:00:00,2016-08-12
5,FLOOD_110040,2016-08-14 06:00:00,2016-08-15
6,FLOOD_119753,2017-08-25 23:45:00,2017-08-28
7,FLOOD_119746,2017-08-26 12:30:00,2017-08-28
8,FLOOD_120357,2017-09-20 00:00:00,2017-09-20
9,FLOOD_196694,2024-09-25 18:00:00,2024-09-26


In [7]:

results = []

market_dates = pd.Index(market_df["Date"])

for _, flood in top_floods.iterrows():
    event_date = flood["event_trading_date"]

    if pd.isna(event_date):
        continue

    idx = market_dates.get_loc(event_date)

    for sector in sectors:
        close_col = f"{sector}_Close"

        for window in windows:
            future_idx = idx + window

            if future_idx >= len(market_df):
                continue

            start_price = market_df.loc[idx, close_col]
            end_price = market_df.loc[future_idx, close_col]

            if pd.isna(start_price) or pd.isna(end_price):
                continue

            if start_price <= 0:
                continue

            pct_return = (
                (end_price / start_price) - 1
            ) * 100

            results.append({
                "flood_id": flood["flood_id"],
                "state": flood["states"],
                "start_date": flood["start_date"],
                "damage_usd": flood["total_damage_usd"],
                "sector": sector,
                "window": window,
                "return_pct": pct_return
            })

flood_returns = pd.DataFrame(results)

display(flood_returns.head(15))


,flood_id,state,start_date,damage_usd,sector,window,return_pct
0,FLOOD_37606,TENNESSEE,2010-05-01 10:00:00,1.569464e+09,SPY,1,-2.351493
1,FLOOD_37606,TENNESSEE,2010-05-01 10:00:00,1.569464e+09,SPY,5,-3.481554
2,FLOOD_37606,TENNESSEE,2010-05-01 10:00:00,1.569464e+09,SPY,14,-9.339420
3,FLOOD_37606,TENNESSEE,2010-05-01 10:00:00,1.569464e+09,XLE,1,-2.673695
4,FLOOD_37606,TENNESSEE,2010-05-01 10:00:00,1.569464e+09,XLE,5,-5.363896
5,FLOOD_37606,TENNESSEE,2010-05-01 10:00:00,1.569464e+09,XLE,14,-12.411290
6,FLOOD_37606,TENNESSEE,2010-05-01 10:00:00,1.569464e+09,XLU,1,-1.719076
7,FLOOD_37606,TENNESSEE,2010-05-01 10:00:00,1.569464e+09,XLU,5,-2.692192
8,FLOOD_37606,TENNESSEE,2010-05-01 10:00:00,1.569464e+09,XLU,14,-7.168353
9,FLOOD_37606,TENNESSEE,2010-05-01 10:00:00,1.569464e+09,XLF,1,-2.732228
